# Day 34 — 평가 해석: accuracy 너머, 그리고 오류 분석

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-5/day-34.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-5/day-34.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 손으로 세는 confusion matrix: accuracy가 속이는 경우

데이터 없이 숫자 몇 개로 시작합니다. 측정 100개 중 90개가 PD군, 10개가 대조군(HC)이고, 모델은 무조건 "PD"라고만 답한다고 합시다.

In [ ]:
import numpy as np
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, precision_score, recall_score)

y_true = np.array([1] * 90 + [0] * 10)     # 실제: PD 90개, HC 10개
y_pred = np.ones(100, dtype=int)           # 모델: 전부 PD라고 답함

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
print(f"TP={tp}  FN={fn}  FP={fp}  TN={tn}")
print(f"accuracy            = {accuracy_score(y_true, y_pred):.2f}")
print(f"recall (PD)         = {recall_score(y_true, y_pred):.2f}   # TP / (TP + FN)")
print(f"precision (PD)      = {precision_score(y_true, y_pred):.2f}   # TP / (TP + FP)")
print(f"recall (HC)         = {recall_score(y_true, y_pred, pos_label=0):.2f}   # TN / (TN + FP), 특이도")
print(f"balanced accuracy   = {balanced_accuracy_score(y_true, y_pred):.2f}   # 두 군 recall의 평균")

### 예시 2 — 참가자 단위 교차검증으로 성능표 만들기

이제 가상 feature table로 돌아옵니다. `cross_val_predict`는 교차검증의 각 fold에서 **평가용이었을 때의 예측**만 모아 줍니다. 300개 측정 모두에 대해 "그 참가자를 학습에 쓰지 않은 모델의 예측"이 하나씩 생깁니다.

In [ ]:
import numpy as np
import pandas as pd

FEATURES = ["stride_time_cv", "accel_rms", "tremor_band_power",
            "jerk_mean", "step_regularity", "arm_swing_asym"]

def make_demo_table(seed=42, n_participants=50, n_repeats=6):
    """연습용 가상 feature table. 실제 환자 데이터가 아니다."""
    rng = np.random.default_rng(seed)
    pd_shift = np.array([0.6, -0.5, 0.6, 0.2, -0.4, 0.3])   # PD군 평균이 이동하는 정도
    rows = []
    for i in range(n_participants):
        group = "PD" if i < n_participants // 2 else "HC"
        age = rng.normal(68 if group == "PD" else 62, 7)
        sex = rng.choice(["F", "M"])
        person = rng.normal(0, 1.0, size=len(FEATURES))      # 그 사람 고유의 '움직임 지문'
        for rep in range(1, n_repeats + 1):
            values = person + rng.normal(0, 0.25, size=len(FEATURES))  # 측정마다 생기는 흔들림
            if group == "PD":
                values = values + pd_shift
            values[0] += 0.06 * (age - 65)                   # stride_time_cv는 나이와도 관련
            rows.append({"participant_id": f"P{i + 1:03d}", "group": group,
                         "age": round(age, 1), "sex": sex, "repeat": rep,
                         **dict(zip(FEATURES, values.round(3)))})
    return pd.DataFrame(rows)

df = make_demo_table()

# 🔁 실제 데이터로 바꿀 지점 ---------------------------------------------
# 4주차(Day 26)에 만든 feature table이 있으면 위의 df = make_demo_table() 대신:
# df = pd.read_csv("data/physionet_pd/derived/pd_features_records.csv")   # 내 파일 경로로
# FEATURES = ["내_feature_1", "내_feature_2"]                # 내 feature 열 이름으로
# 그리고 아래 두 이름을 내 표의 열 이름에 맞춘다.
ID_COL = "participant_id"   # 참가자 ID 열
GROUP_COL = "group"         # 진단군 열 (이 예시에서는 "PD" / "HC")
# -------------------------------------------------------------------------

from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
                             average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = df[FEATURES]
y = (df[GROUP_COL] == "PD").astype(int)       # PD = 1 (positive), HC = 0   🔁 내 군 이름으로
groups = df[ID_COL]
cv = GroupKFold(n_splits=5)                   # Day 33과 같은 참가자 단위 분할

baseline = make_pipeline(StandardScaler(), LogisticRegression())
dummy = DummyClassifier(strategy="most_frequent")

def out_of_fold_proba(model):
    """각 측정이 평가용 fold에 있었을 때의 'PD군일 확률'."""
    return cross_val_predict(model, X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]

def metric_row(proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {"accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred),
            "f1": f1_score(y, pred, zero_division=0),
            "roc_auc": roc_auc_score(y, proba),
            "pr_auc": average_precision_score(y, proba)}

proba = out_of_fold_proba(baseline)
pred = (proba >= 0.5).astype(int)

table = pd.DataFrame({"dummy": metric_row(out_of_fold_proba(dummy)),
                      "logistic_baseline": metric_row(proba)}).round(2)
print("confusion matrix (행 = 실제 HC/PD, 열 = 예측 HC/PD):")
print(confusion_matrix(y, pred))
print()
print(table)
print(f"\nPD군 측정의 비율(PR-AUC의 기준선) = {y.mean():.2f}")

# fold마다 ROC-AUC가 얼마나 다른지
fold_auc = [roc_auc_score(y.iloc[test], proba[test]) for _, test in cv.split(X, y, groups)]
print("fold별 ROC-AUC:", np.round(fold_auc, 2), f"→ 범위 {min(fold_auc):.2f}–{max(fold_auc):.2f}")

Path("reports").mkdir(exist_ok=True)
table.to_csv("reports/day34_performance_table.csv")

Path("figures").mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
ConfusionMatrixDisplay.from_predictions(y, pred, display_labels=["HC", "PD"], cmap="Blues",
                                        colorbar=False, ax=axes[0])
axes[0].set_title("Out-of-fold confusion matrix (rows = recordings)")
RocCurveDisplay.from_predictions(y, proba, name="logistic baseline", plot_chance_level=True, ax=axes[1])
axes[1].set_title("ROC curve (participant-wise 5-fold CV)")
fig.suptitle("Synthetic demo data — baseline separating PD vs HC within this dataset")
fig.tight_layout()
fig.savefig("figures/day34_confusion_roc.png", dpi=150)
plt.show()

### 예시 3 — 오류 분석: 누구를 틀렸는가

예시 2에 이어서 실행합니다. 측정 단위 예측을 참가자 단위로 모읍니다.

In [ ]:
errors = df[[ID_COL, GROUP_COL, "age"]].assign(proba=proba, wrong=(pred != y))

by_person = (errors.groupby([ID_COL, GROUP_COL])
             .agg(age=("age", "first"),
                  mean_proba=("proba", "mean"),        # 측정 6개의 평균 'PD군일 확률'
                  wrong_frac=("wrong", "mean"))        # 틀린 측정의 비율
             .reset_index())
by_person["mostly_wrong"] = by_person["wrong_frac"] > 0.5

n_all_right = (by_person["wrong_frac"] == 0).sum()
n_all_wrong = (by_person["wrong_frac"] == 1).sum()
print(f"측정을 전부 맞힌 참가자 {n_all_right}명, 전부 틀린 참가자 {n_all_wrong}명, "
      f"섞인 참가자 {len(by_person) - n_all_right - n_all_wrong}명")
print()
print("가장 많이 틀린 참가자 6명:")
print(by_person.sort_values("wrong_frac", ascending=False).head(6).round(2).to_string(index=False))
print()
print("군별 · 오류 여부별 인원과 평균 나이:")
print(by_person.groupby([GROUP_COL, "mostly_wrong"]).agg(n=("age", "size"), mean_age=("age", "mean")).round(1))